# 04 — Classical ML Models on v1.0-trainval Subset (5-Fold CV)
## Multi-Attribute Scene Classification on nuScenes

**Purpose:** Train and evaluate 5 classical models on each of 4 target attributes using the 150-scene subset under **5-fold scene-aware cross-validation**, matching the Stage 1 (v1.0-mini) protocol so the two stages are directly comparable.

> **This notebook replaces both the old single-split `04` and the `04b_patch_xgb_mlp` patch.** The LabelEncoder fix that `04b` applied to XGBoost/MLP is now baked in for all models, so there is no separate patch step.

### Models (identical to Stage 1)
1. Logistic Regression  2. SVM (RBF)  3. Random Forest  4. XGBoost  5. MLP (1 hidden layer)
Plus 2 baselines: random (uniform) and majority-class.

### Experimental design
| Dimension | Value |
|---|---|
| Models | 5 + 2 baselines |
| Attributes | 4 |
| Versions | 2 (base + tuned) |
| Seeds | 3 (42, 7, 123) |
| Outer CV | **5-fold scene-aware** (from notebook 03) |
| **Total fits** | **5 × 4 × 2 × 3 × 5 = 600** |
| Tuning | inner 3-fold StratifiedKFold on each outer training set |

### What changed from the old single-split 04
- Single 70/15/15 split → 5-fold scene-aware CV (test = fold *k*, train = the rest).
- Aggregation is now mean ± std **across folds and seeds** (not seeds only).
- `LabelEncoder` baked in for every model (kills the XGBoost "Invalid classes" and MLP "isnan" errors). Models saved as `{'model', 'label_encoder'}`.
- AUC computed on encoded integer labels aligned to the probability columns (robust).
- Single-class (attribute, fold) combos are skipped and documented.

### Expected runtime
~6–12 hours unattended (SVM with `probability=True` on 6,021 keyframes is the bottleneck). Run overnight.

### Inputs
- `data/processed/v1.0-trainval/features/features_full.csv` (from 02)
- `data/processed/v1.0-trainval/labels/attribute_labels.csv` (from 01)
- `data/processed/v1.0-trainval/splits/sample_fold_assignments.csv` (from 03)
- `data/processed/v1.0-trainval/splits/kfold_metadata.json` (from 03)

### Outputs
- `results/v1.0-trainval/metrics/all_metrics.csv` (per-fit, includes `fold`)
- `results/v1.0-trainval/metrics/baseline_metrics.csv` (per-fold)
- `results/v1.0-trainval/metrics/skipped_combos.json`
- `results/v1.0-trainval/predictions/predictions_test.csv` (includes `fold`)
- `models/v1.0-trainval/<attr>/<model>_<version>_seed<seed>_fold<k>.pkl`
- `results/v1.0-trainval/final/training_summary.json`


## 0. Setup

In [1]:
import os
import json
import time
import joblib
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
print('Imports OK')

Imports OK


## 0.1 Locate Project Root

In [2]:
def find_project_root():
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if (candidate / 'README.md').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError(f'Could not find project root from {p}')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\leemi\Documents\GitHub\nuscenes-scene-classification-ml


## 1. Configuration

In [3]:
DATASET_VERSION = 'v1.0-trainval'

# Paths
PROCESSED_DIR = Path('data/processed') / DATASET_VERSION
FEATURES_PATH = PROCESSED_DIR / 'features' / 'features_full.csv'
LABELS_PATH   = PROCESSED_DIR / 'labels' / 'attribute_labels.csv'
FOLDS_PATH    = PROCESSED_DIR / 'splits' / 'sample_fold_assignments.csv'
KFOLD_META    = PROCESSED_DIR / 'splits' / 'kfold_metadata.json'

RESULTS_DIR   = Path('results') / DATASET_VERSION
METRICS_DIR   = RESULTS_DIR / 'metrics'
PRED_DIR      = RESULTS_DIR / 'predictions'
FINAL_DIR     = RESULTS_DIR / 'final'
MODELS_DIR    = Path('models') / DATASET_VERSION

for d in [METRICS_DIR, PRED_DIR, FINAL_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Experiment design (matches Stage 1)
ATTRIBUTES = ['time_of_day', 'weather', 'vehicle_density', 'vru_present']
CLASS_ORDERS = {
    'time_of_day':     ['day', 'night'],
    'weather':         ['clear', 'rain'],
    'vehicle_density': ['low', 'medium', 'high'],
    'vru_present':     ['absent', 'present'],
}
MODEL_NAMES = ['LogisticRegression', 'SVM_RBF', 'RandomForest', 'XGBoost', 'MLP']
SEED_LIST = [42, 7, 123]
INNER_CV  = 3   # inner grid-search folds

# Number of outer folds — read from notebook 03's metadata
with open(KFOLD_META) as f:
    N_FOLDS = json.load(f)['n_folds']

# Save every fitted model (600 files). Set False if disk-constrained:
# the predictions CSV retains all info needed for metrics and the comparison.
SAVE_MODELS = True

print(f'DATASET_VERSION = {DATASET_VERSION}')
print(f'N_FOLDS         = {N_FOLDS}')
print(f'SEED_LIST       = {SEED_LIST}')
print(f'Total model fits = {len(ATTRIBUTES)*len(MODEL_NAMES)*2*len(SEED_LIST)*N_FOLDS}')

DATASET_VERSION = v1.0-trainval
N_FOLDS         = 5
SEED_LIST       = [42, 7, 123]
Total model fits = 600


## 2. Load Data

In [4]:
print('Loading features (this may take 30-60 sec for the 6,021 x 6,216 matrix)...')
t0 = time.time()
df_feat = pd.read_csv(FEATURES_PATH)
print(f'Features loaded in {time.time()-t0:.1f} sec: shape {df_feat.shape}')

df_labels = pd.read_csv(LABELS_PATH)
df_folds  = pd.read_csv(FOLDS_PATH)
print(f'Labels: {df_labels.shape}   Folds: {df_folds.shape}')

# Merge into single dataframe via sample_token
df = df_feat.merge(df_labels[['sample_token'] + ATTRIBUTES], on='sample_token')
df = df.merge(df_folds[['sample_token', 'fold']], on='sample_token')
print(f'Merged: {df.shape}')

# Feature columns
feature_cols = [c for c in df.columns if c.startswith(('hog_', 'color_', 'lbp_', 'photo_'))]
print(f'Feature columns: {len(feature_cols)}')
assert df['fold'].notna().all(), 'Some keyframes have no fold assignment'
print(f'Folds present: {sorted(df["fold"].unique())}')

Loading features (this may take 30-60 sec for the 6,021 x 6,216 matrix)...
Features loaded in 2.6 sec: shape (6021, 6217)
Labels: (6021, 12)   Folds: (6021, 7)
Merged: (6021, 6222)
Feature columns: 6216
Folds present: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


## 3. Model Factories and Grids (identical to Stage 1)

In [5]:
def make_base_model(name, seed):
    """Default-parameter version of each model."""
    if name == 'LogisticRegression':
        return Pipeline([('scaler', StandardScaler()),
                         ('clf', LogisticRegression(class_weight='balanced',
                                                    max_iter=2000, random_state=seed))])
    elif name == 'SVM_RBF':
        return Pipeline([('scaler', StandardScaler()),
                         ('clf', SVC(kernel='rbf', class_weight='balanced',
                                     probability=True, random_state=seed))])
    elif name == 'RandomForest':
        return RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                      random_state=seed, n_jobs=-1)
    elif name == 'XGBoost':
        return XGBClassifier(n_estimators=200, learning_rate=0.1,
                             eval_metric='mlogloss', random_state=seed,
                             n_jobs=-1, verbosity=0)
    elif name == 'MLP':
        return Pipeline([('scaler', StandardScaler()),
                         ('clf', MLPClassifier(hidden_layer_sizes=(128,), max_iter=300,
                                               random_state=seed, early_stopping=True,
                                               n_iter_no_change=10))])
    raise ValueError(name)

def get_tuned_grid(name):
    """Grid-search space for each model (identical to Stage 1)."""
    if name == 'LogisticRegression':
        return {'clf__C': [0.01, 0.1, 1.0, 10.0]}
    elif name == 'SVM_RBF':
        return {'clf__C': [0.1, 1.0, 10.0], 'clf__gamma': ['scale', 0.01, 0.1]}
    elif name == 'RandomForest':
        return {'max_depth': [None, 10, 30], 'min_samples_split': [2, 5, 10]}
    elif name == 'XGBoost':
        return {'max_depth': [3, 6, 9], 'learning_rate': [0.05, 0.1, 0.2]}
    elif name == 'MLP':
        return {'clf__hidden_layer_sizes': [(64,), (128,), (256,)],
                'clf__alpha': [1e-4, 1e-3]}
    raise ValueError(name)

print(f'Model factories defined for {len(MODEL_NAMES)} models')

Model factories defined for 5 models


## 4. Metric Helper

Classification metrics are computed on string labels; AUC is computed on encoded integers aligned to the probability columns (robust to class naming).

In [6]:
def compute_metrics(y_true_str, y_pred_str, y_true_int=None, y_proba=None, n_classes=None):
    out = {
        'accuracy':        accuracy_score(y_true_str, y_pred_str),
        'macro_f1':        f1_score(y_true_str, y_pred_str, average='macro', zero_division=0),
        'weighted_f1':     f1_score(y_true_str, y_pred_str, average='weighted', zero_division=0),
        'macro_precision': precision_score(y_true_str, y_pred_str, average='macro', zero_division=0),
        'macro_recall':    recall_score(y_true_str, y_pred_str, average='macro', zero_division=0),
    }
    out['auc'] = np.nan
    if y_proba is not None and y_true_int is not None and n_classes is not None:
        try:
            if n_classes == 2:
                out['auc'] = roc_auc_score(y_true_int, y_proba[:, 1])
            else:
                out['auc'] = roc_auc_score(y_true_int, y_proba, multi_class='ovr',
                                           average='macro', labels=list(range(n_classes)))
        except Exception:
            out['auc'] = np.nan
    return out

print('Metric helper defined')

Metric helper defined


## 5. Baselines (random + majority), per fold

In [7]:
baseline_rows = []
for attr in ATTRIBUTES:
    classes = CLASS_ORDERS[attr]
    for fold in range(N_FOLDS):
        train_mask = df['fold'] != fold
        test_mask  = df['fold'] == fold
        y_train = df.loc[train_mask, attr].values
        y_test  = df.loc[test_mask,  attr].values
        # Skip same single-class folds the models will skip
        if pd.Series(y_train).nunique() < 2 or pd.Series(y_test).nunique() < 2:
            continue
        majority_class = pd.Series(y_train).value_counts().idxmax()
        for seed in SEED_LIST:
            rng = np.random.default_rng(seed)
            y_rand = rng.choice(classes, size=len(y_test))
            m = compute_metrics(y_test, y_rand)
            baseline_rows.append({**m, 'attribute': attr, 'baseline': 'random',
                                  'seed': seed, 'fold': fold})
            y_maj = np.full(len(y_test), majority_class)
            m = compute_metrics(y_test, y_maj)
            baseline_rows.append({**m, 'attribute': attr, 'baseline': 'majority',
                                  'seed': seed, 'fold': fold})

df_baselines = pd.DataFrame(baseline_rows)
df_baselines.to_csv(METRICS_DIR / 'baseline_metrics.csv', index=False)
print(f'Saved -> {METRICS_DIR / "baseline_metrics.csv"}  ({len(df_baselines)} rows)')
print('\n=== BASELINE PERFORMANCE (test, mean across folds+seeds) ===')
print(df_baselines.groupby(['attribute', 'baseline'])[['accuracy', 'macro_f1']].mean().round(3))

Saved -> results\v1.0-trainval\metrics\baseline_metrics.csv  (120 rows)

=== BASELINE PERFORMANCE (test, mean across folds+seeds) ===
                          accuracy  macro_f1
attribute       baseline                    
time_of_day     majority     0.800     0.444
                random       0.501     0.452
vehicle_density majority     0.338     0.168
                random       0.337     0.330
vru_present     majority     0.532     0.346
                random       0.500     0.498
weather         majority     0.799     0.443
                random       0.501     0.446


## 6. Train Function (5-fold, LabelEncoder baked in)

For outer fold *k*: test = keyframes with `fold == k`, train = all other folds. Tuning uses inner 3-fold CV on the outer training set.

In [8]:
def fit_and_evaluate(attr, model_name, version, seed, fold):
    classes   = CLASS_ORDERS[attr]
    n_classes = len(classes)

    train_mask = df['fold'] != fold
    test_mask  = df['fold'] == fold
    X_train = df.loc[train_mask, feature_cols].values
    X_test  = df.loc[test_mask,  feature_cols].values
    y_train_str = df.loc[train_mask, attr].values
    y_test_str  = df.loc[test_mask,  attr].values

    # Stable string -> int mapping from CLASS_ORDERS
    le = LabelEncoder()
    le.fit(classes)
    y_train = le.transform(y_train_str)
    y_test  = le.transform(y_test_str)

    base_model = make_base_model(model_name, seed)

    if version == 'base':
        model = base_model
        model.fit(X_train, y_train)
        best_params = None
    else:  # tuned
        grid = get_tuned_grid(model_name)
        min_class_count = int(pd.Series(y_train).value_counts().min())
        actual_cv = min(INNER_CV, min_class_count)
        if actual_cv < 2:
            model = base_model
            model.fit(X_train, y_train)
            best_params = 'tuning_skipped_minority_class_too_small'
        else:
            cv = StratifiedKFold(n_splits=actual_cv, shuffle=True, random_state=seed)
            search = GridSearchCV(base_model, grid, cv=cv, scoring='f1_macro', n_jobs=-1)
            search.fit(X_train, y_train)
            model = search.best_estimator_
            best_params = search.best_params_

    # Predict (model works in int space); decode predictions back to strings
    y_pred_int = model.predict(X_test)
    y_pred_str = le.inverse_transform(y_pred_int)
    try:
        y_proba = model.predict_proba(X_test)
    except Exception:
        y_proba = None

    m = compute_metrics(y_test_str, y_pred_str, y_test, y_proba, n_classes)
    m.update({'attribute': attr, 'model': model_name, 'version': version,
              'seed': seed, 'fold': fold,
              'best_params': str(best_params) if best_params else ''})

    if SAVE_MODELS:
        model_path = MODELS_DIR / attr / f'{model_name}_{version}_seed{seed}_fold{fold}.pkl'
        model_path.parent.mkdir(parents=True, exist_ok=True)
        joblib.dump({'model': model, 'label_encoder': le}, model_path)

    sample_tokens_test = df.loc[test_mask, 'sample_token'].values
    pred_rows = [{'sample_token': st, 'attribute': attr, 'model': model_name,
                  'version': version, 'seed': seed, 'fold': fold,
                  'y_true': yt, 'y_pred': yp}
                 for st, yt, yp in zip(sample_tokens_test, y_test_str, y_pred_str)]
    return m, pred_rows

print('Training function defined')

Training function defined


## 7. Main Training Loop (5 × 4 × 2 × 3 × folds)

In [ ]:
print('=' * 70)
print('STARTING 5-FOLD TRAINING LOOP')
print('=' * 70)
total_fits = len(ATTRIBUTES) * len(MODEL_NAMES) * 2 * len(SEED_LIST) * N_FOLDS
print(f'Max fits: {total_fits}   (minus any single-class folds skipped)')
print(f'Started at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('=' * 70)

all_metrics, all_predictions, skipped_combos = [], [], []
t_global_start = time.time()
fit_count = 0

for attr in ATTRIBUTES:
    for fold in range(N_FOLDS):
        # Skip-and-document single-class (attribute, fold) combos
        n_train_cls = df.loc[df['fold'] != fold, attr].nunique()
        n_test_cls  = df.loc[df['fold'] == fold, attr].nunique()
        if n_train_cls < 2 or n_test_cls < 2:
            skipped_combos.append({'attribute': attr, 'fold': fold,
                                   'n_train_classes': int(n_train_cls),
                                   'n_test_classes': int(n_test_cls)})
            print(f'  [SKIP] {attr:18s} fold {fold}: '
                  f'train_classes={n_train_cls}, test_classes={n_test_cls}')
            continue
        for model_name in MODEL_NAMES:
            for version in ['base', 'tuned']:
                for seed in SEED_LIST:
                    fit_count += 1
                    t0 = time.time()
                    try:
                        m, preds = fit_and_evaluate(attr, model_name, version, seed, fold)
                        all_metrics.append(m)
                        all_predictions.extend(preds)
                        print(f'[{fit_count:3d}/{total_fits}] {attr:18s} {model_name:18s} '
                              f'{version:5s} seed={seed:3d} fold={fold} '
                              f'macro_f1={m["macro_f1"]:.3f}  t={time.time()-t0:5.1f}s')
                    except Exception as e:
                        print(f'[{fit_count:3d}/{total_fits}] {attr:18s} {model_name:18s} '
                              f'{version:5s} seed={seed:3d} fold={fold} [FAIL] {str(e)[:70]}')

t_elapsed = time.time() - t_global_start
print('=' * 70)
print(f'TRAINING COMPLETE - {t_elapsed/60:.1f} min, '
      f'{len(all_metrics)} fits, {len(skipped_combos)} skipped combos')
print('=' * 70)

STARTING 5-FOLD TRAINING LOOP
Max fits: 600   (minus any single-class folds skipped)
Started at: 2026-05-23 06:01:16
[  1/600] time_of_day        LogisticRegression base  seed= 42 fold=0 macro_f1=0.989  t=  1.0s
[  2/600] time_of_day        LogisticRegression base  seed=  7 fold=0 macro_f1=0.989  t=  1.0s
[  3/600] time_of_day        LogisticRegression base  seed=123 fold=0 macro_f1=0.989  t=  1.1s
[  4/600] time_of_day        LogisticRegression tuned seed= 42 fold=0 macro_f1=0.979  t=  5.6s
[  5/600] time_of_day        LogisticRegression tuned seed=  7 fold=0 macro_f1=0.989  t=  5.5s
[  6/600] time_of_day        LogisticRegression tuned seed=123 fold=0 macro_f1=0.999  t=  4.8s
[  7/600] time_of_day        SVM_RBF            base  seed= 42 fold=0 macro_f1=0.963  t= 35.0s
[  8/600] time_of_day        SVM_RBF            base  seed=  7 fold=0 macro_f1=0.963  t= 34.8s
[  9/600] time_of_day        SVM_RBF            base  seed=123 fold=0 macro_f1=0.963  t= 34.0s
[ 10/600] time_of_day       

In [3]:
import os
import json
import pandas as pd
from pathlib import Path

def find_project_root():
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if (candidate / 'README.md').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('Project root not found')

os.chdir(find_project_root())   # <-- this is what was missing

df = pd.read_csv('results/v1.0-trainval/metrics/all_metrics.csv')
skipped = json.load(open('results/v1.0-trainval/metrics/skipped_combos.json'))

print('='*70)
print('TRAINING COMPLETE (summary regenerated from saved results)')
print('='*70)
print(f'Total fits saved : {len(df)}')
print(f'Attributes       : {sorted(df.attribute.unique())}')
print(f'Models           : {sorted(df.model.unique())}')
print(f'Folds            : {sorted(df.fold.unique())}')
print(f'Seeds            : {sorted(df.seed.unique())}')
print(f'Skipped combos   : {len(skipped)}  {skipped}')
print()
print('Last fit recorded:', df.iloc[-1][['attribute','model','version','seed','fold']].to_dict())
print()
print('Best tuned macro-F1 per attribute (mean across folds+seeds):')
best = df[df.version=='tuned'].groupby(['attribute','model']).macro_f1.mean().reset_index()
for a in df.attribute.unique():
    top = best[best.attribute==a].sort_values('macro_f1').iloc[-1]
    print(f'  {a:18s}: {top.model:18s} {top.macro_f1:.3f}')

TRAINING COMPLETE (summary regenerated from saved results)
Total fits saved : 600
Attributes       : ['time_of_day', 'vehicle_density', 'vru_present', 'weather']
Models           : ['LogisticRegression', 'MLP', 'RandomForest', 'SVM_RBF', 'XGBoost']
Folds            : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Seeds            : [np.int64(7), np.int64(42), np.int64(123)]
Skipped combos   : 0  []

Last fit recorded: {'attribute': 'vru_present', 'model': 'MLP', 'version': 'tuned', 'seed': 123, 'fold': 4}

Best tuned macro-F1 per attribute (mean across folds+seeds):
  time_of_day       : XGBoost            0.993
  weather           : RandomForest       0.901
  vehicle_density   : SVM_RBF            0.511
  vru_present       : SVM_RBF            0.628


## 8. Save Metrics, Predictions, and Skipped-Combos Manifest

In [8]:
import os, json
import pandas as pd
from pathlib import Path

def find_project_root():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c/'README.md').exists() and (c/'notebooks').exists(): return c
    raise FileNotFoundError('Project root not found')
os.chdir(find_project_root())

METRICS_DIR = Path('results/v1.0-trainval/metrics')
PRED_DIR    = Path('results/v1.0-trainval/predictions')

# Results were saved by the training loop. Here we confirm they exist (no re-save needed).
df_metrics  = pd.read_csv(METRICS_DIR / 'all_metrics.csv')
df_preds    = pd.read_csv(PRED_DIR / 'predictions_test.csv')
skipped_combos = json.load(open(METRICS_DIR / 'skipped_combos.json'))

print(f'Loaded all_metrics.csv      ({len(df_metrics)} rows)')
print(f'Loaded predictions_test.csv ({len(df_preds)} rows)')
print(f'Loaded skipped_combos.json  ({len(skipped_combos)} combos)')

Loaded all_metrics.csv      (600 rows)
Loaded predictions_test.csv (722520 rows)
Loaded skipped_combos.json  (0 combos)


## 9. Headline Summary (mean macro-F1 across folds + seeds)

In [9]:
import pandas as pd
from pathlib import Path

METRICS_DIR = Path('results/v1.0-trainval/metrics')
df_metrics   = pd.read_csv(METRICS_DIR / 'all_metrics.csv')
df_baselines = pd.read_csv(METRICS_DIR / 'baseline_metrics.csv')

print('=' * 70)
print('HEADLINE RESULTS (tuned, mean +/- std across folds and seeds)')
print('=' * 70)
headline = (df_metrics[df_metrics['version'] == 'tuned']
            .groupby(['attribute', 'model'])['macro_f1']
            .agg(['mean', 'std']).round(3))
print(headline)

best_per_attr = (df_metrics[df_metrics['version'] == 'tuned']
                 .groupby(['attribute', 'model'])['macro_f1'].mean().reset_index())
best_per_attr = best_per_attr.loc[best_per_attr.groupby('attribute')['macro_f1'].idxmax()]
print('\n=== BEST MODEL PER ATTRIBUTE (Stage 2, 5-fold) ===')
for _, r in best_per_attr.iterrows():
    base_rand = df_baselines[(df_baselines['attribute'] == r['attribute']) &
                             (df_baselines['baseline'] == 'random')]['macro_f1'].mean()
    verdict = 'above random' if r['macro_f1'] > base_rand else 'below random'
    print(f'  {r["attribute"]:18s}: {r["model"]:18s} macro_f1={r["macro_f1"]:.3f}  '
          f'(random={base_rand:.3f}, {verdict})')

HEADLINE RESULTS (tuned, mean +/- std across folds and seeds)
                                     mean    std
attribute       model                           
time_of_day     LogisticRegression  0.993  0.007
                MLP                 0.969  0.027
                RandomForest        0.976  0.031
                SVM_RBF             0.978  0.026
                XGBoost             0.993  0.012
vehicle_density LogisticRegression  0.486  0.036
                MLP                 0.489  0.017
                RandomForest        0.442  0.034
                SVM_RBF             0.511  0.022
                XGBoost             0.506  0.016
vru_present     LogisticRegression  0.601  0.036
                MLP                 0.615  0.026
                RandomForest        0.599  0.060
                SVM_RBF             0.628  0.033
                XGBoost             0.617  0.028
weather         LogisticRegression  0.858  0.055
                MLP                 0.850  0.067
       

## 10. Save Training Summary

In [10]:
import json
from datetime import datetime
import pandas as pd
from pathlib import Path

METRICS_DIR = Path('results/v1.0-trainval/metrics')
FINAL_DIR   = Path('results/v1.0-trainval/final'); FINAL_DIR.mkdir(parents=True, exist_ok=True)

df_metrics  = pd.read_csv(METRICS_DIR / 'all_metrics.csv')
df_preds    = pd.read_csv('results/v1.0-trainval/predictions/predictions_test.csv')
skipped_combos = json.load(open(METRICS_DIR / 'skipped_combos.json'))

DATASET_VERSION = 'v1.0-trainval'
N_FOLDS = int(df_metrics['fold'].nunique())

best_per_attr = (df_metrics[df_metrics['version'] == 'tuned']
                 .groupby(['attribute', 'model'])['macro_f1'].mean().reset_index())
best_per_attr = best_per_attr.loc[best_per_attr.groupby('attribute')['macro_f1'].idxmax()]

summary = {
    'dataset_version':   DATASET_VERSION,
    'split_method':      f'{N_FOLDS}-fold scene-aware CV',
    'summary_generated': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'n_fits':            len(df_metrics),
    'n_predictions':     len(df_preds),
    'n_skipped_combos':  len(skipped_combos),
    'skipped_combos':    skipped_combos,
    'best_per_attribute': {
        r['attribute']: {'model': r['model'], 'macro_f1': round(r['macro_f1'], 3)}
        for _, r in best_per_attr.iterrows()
    },
}
with open(FINAL_DIR / 'training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Saved -> {FINAL_DIR / "training_summary.json"}')
print(json.dumps(summary, indent=2))

Saved -> results\v1.0-trainval\final\training_summary.json
{
  "dataset_version": "v1.0-trainval",
  "split_method": "5-fold scene-aware CV",
  "summary_generated": "2026-05-25 01:10:54",
  "n_fits": 600,
  "n_predictions": 722520,
  "n_skipped_combos": 0,
  "skipped_combos": [],
  "best_per_attribute": {
    "time_of_day": {
      "model": "XGBoost",
      "macro_f1": 0.993
    },
    "vehicle_density": {
      "model": "SVM_RBF",
      "macro_f1": 0.511
    },
    "vru_present": {
      "model": "SVM_RBF",
      "macro_f1": 0.628
    },
    "weather": {
      "model": "RandomForest",
      "macro_f1": 0.901
    }
  }
}


## 11. Next Steps

In [ ]:
import json
import pandas as pd
from pathlib import Path

METRICS_DIR = Path('results/v1.0-trainval/metrics')
df_metrics = pd.read_csv(METRICS_DIR / 'all_metrics.csv')
skipped_combos = json.load(open(METRICS_DIR / 'skipped_combos.json'))
N_FOLDS = int(df_metrics['fold'].nunique())

print('=' * 70)
print('STAGE 2 MODEL TRAINING COMPLETE (5-fold)')
print('=' * 70)
print(f'Fits: {len(df_metrics)}   Skipped: {len(skipped_combos)}')
print()
print('Next: 05_compare_mini_vs_subset.ipynb')
print('  - Both stages 5-fold -> like-for-like comparison')
print('  - Stage 2 metrics carry a "fold" column; confusion matrices aggregate over folds')
print('  - Cross-stage comparison reported descriptively (different datasets)')

STAGE 2 MODEL TRAINING COMPLETE (5-fold)


NameError: name 'df_metrics' is not defined